# Post-VCF analysis of small variants
*INRAe-IFB-INSERM Bioinformatics School 2026*

* Elodie Girard - Institut Curie

<style>
/* Slide text size and vertical overflow */
.reveal { font-size: 26px !important; }
.reveal .slides section.stack { overflow: visible !important; max-height: none !important; }
.reveal .slides section:not(.stack) { max-height: 100% !important; overflow-y: auto !important; }
.reveal img { max-height: 255px; width: auto; }
/* Fenced markdown blocks */
.reveal pre code { font-size: 1.2em !important; line-height: 1.4 !important; }
/* Code cell outputs */
.jp-OutputArea-output, .jp-OutputArea-output pre {
    font-size: 18px !important; max-height: 290px !important;
    overflow-y: auto !important; overflow-x: auto !important;
    white-space: pre !important; width: 100% !important; box-sizing: border-box !important;
}
.jp-OutputArea-output table, .jp-OutputArea-output table th, .jp-OutputArea-output table td { font-size: 16px !important; }
/* Code cell inputs */
.jp-InputArea .highlight { overflow-x: auto !important; max-width: 100%; display: block; }
.jp-InputArea .highlight pre { font-size: 18px !important; white-space: pre !important; }
/* Very tall outputs (cell tag tall_output, exported as celltag_tall_output): bounded box instead of overflowing the slide */
.celltag_tall_output .jp-OutputArea-output { display: block !important; max-height: 290px !important; overflow-y: auto !important; }
</style>

Check that you are still in your `atelier_variant/TP_small_variants` folder, and load the packages used in this notebook.

In [1]:
suppressPackageStartupMessages({
  library(tibble)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(purrr)
  library(ggplot2)
})

## I - Where are we in the workflow?

**From reads to an annotated VCF** (previous notebooks)

<div style="display:flex; flex-wrap:wrap; align-items:stretch; gap:6px; font-size:0.85em;">
<div style="background:#f0f2f5; border:2px solid #9aa5b1; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Reads</strong><br><span style="font-size:0.9em">FASTQ<br>+ reference genome (FASTA)</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#f0f2f5; border:2px solid #9aa5b1; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Alignment</strong><br><span style="font-size:0.9em">BWA-MEM2<br>➜ BAM</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#f0f2f5; border:2px solid #9aa5b1; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Post-alignment processing<br>&amp; QC</strong><br><span style="font-size:0.9em">GATK MarkDuplicates<br>samtools, mosdepth, qualimap</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#dafbe1; border:2px solid #1f883d; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Variant calling</strong><br><span style="font-size:0.9em">GATK HaplotypeCaller<br>➜ VCF</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#dafbe1; border:2px solid #1f883d; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Annotation</strong><br><span style="font-size:0.9em">SnpEff / SnpSift<br>➜ annotated VCF</span></div>
</div>

**Post-VCF analysis** (this notebook)

<div style="display:flex; flex-wrap:wrap; align-items:stretch; gap:6px; font-size:0.85em;">
<div style="background:#f0f2f5; border:2px solid #9aa5b1; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Reads</strong><br><span style="font-size:0.9em">FASTQ<br>+ reference genome (FASTA)</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#f0f2f5; border:2px solid #9aa5b1; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Alignment</strong><br><span style="font-size:0.9em">BWA-MEM2<br>➜ BAM</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#f0f2f5; border:2px solid #9aa5b1; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Post-alignment processing<br>&amp; QC</strong><br><span style="font-size:0.9em">GATK MarkDuplicates<br>samtools, mosdepth, qualimap</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#dafbe1; border:2px solid #1f883d; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Variant calling</strong><br><span style="font-size:0.9em">GATK HaplotypeCaller<br>➜ VCF</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#dafbe1; border:2px solid #1f883d; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Annotation</strong><br><span style="font-size:0.9em">SnpEff / SnpSift<br>➜ annotated VCF</span></div>
</div>

<div style="text-align:center; font-size:1.3em; margin:6px;">⬇</div>

<div style="display:flex; flex-wrap:wrap; gap:6px; font-size:0.85em;">
<div style="background:#e7f3fe; border:2px solid #2196F3; border-radius:8px; padding:8px 10px;"><strong>QC &amp; filters</strong><br>descriptive statistics</div>
<div style="background:#e7f3fe; border:2px solid #2196F3; border-radius:8px; padding:8px 10px;"><strong>Marker / QTL search</strong></div>
<div style="background:#e7f3fe; border:2px solid #2196F3; border-radius:8px; padding:8px 10px;"><strong>GWAS</strong><br>Genome-Wide Association Study</div>
<div style="background:#e7f3fe; border:2px solid #2196F3; border-radius:8px; padding:8px 10px;"><strong>Phylogenetic analysis</strong></div>
<div style="background:#e7f3fe; border:2px solid #2196F3; border-radius:8px; padding:8px 10px;"><strong>...</strong></div>
</div>

**What comes after the VCF depends on the biological question.**

## II - Reminder: the VCF format

A VCF file has three parts:

* **Meta-information** (lines starting with `##`): definitions of the INFO, FORMAT and FILTER fields, contigs, command line...
* **Header line** (starting with a single `#`): the column names.
* **Body**: one line per variant, 8 fixed columns (`CHROM POS ID REF ALT QUAL FILTER INFO`), then `FORMAT`, then **one column per sample**.

The `FORMAT` column describes how to read each sample column (for example `GT:AD:DP:GQ:PL`).

Two real records from our annotated multi-sample VCF (4 animals: 2 carriers of the milca phenotype, 2 controls), embedded here so that the cells run without any file.

In [2]:
vcf_header <- c(
  '##fileformat=VCFv4.2',
  '##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">',
  '##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">',
  '##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth">',
  '##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">',
  '##INFO=<ID=QD,Number=1,Type=Float,Description="Variant Confidence/Quality by Depth">',
  '##INFO=<ID=ANN,...>',
  '##contig=<ID=NC_037329.1,length=136231102>',
  paste(c("#CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT",
          "milca_40995", "milca_40998", "control_41011", "control_41012"), collapse = "\t")
)

**Record 1**: `C>A` in an intron of *TRAK2* (heterozygous in the two controls).

In [3]:
rec_1 <- paste(c("NC_037329.1", "90000329", ".", "C", "A", "226.62", ".",
        paste0("AC=2;AF=0.250;AN=8;BaseQRankSum=0.00;DP=54;ExcessHet=0.6695;FS=0.000;MLEAC=2;MLEAF=0.250;",
               "MQ=60.00;MQRankSum=0.00;QD=6.30;ReadPosRankSum=-1.640e-01;SOR=0.781;",
               "ANN=A|intron_variant|MODIFIER|exon-NM_001206233.1-1|GENE_exon-NM_001206233.1-1|transcript|rna-NM_001206233.1|protein_coding|1/15|c.-198+7962G>T||||||,",
               "A|non_coding_transcript_variant|MODIFIER|TRAK2|gene-TRAK2|transcript|NM_001206233.1|protein_coding||||||||WARNING_TRANSCRIPT_NO_START_CODON"),
        "GT:AD:DP:GQ:PL",
        "0/0:2,0:2:6:0,6,49", "0/0:16,0:16:17:0,17,315", "0/1:10,6:16:99:100,0,199", "0/1:12,8:20:99:138,0,237"),
      collapse = "\t")

**Record 2**: `G>C`, missense in *FZD7* (heterozygous in the two milca animals).

In [4]:
rec_2 <- paste(c("NC_037329.1", "90572569", ".", "G", "C", "612.00", ".",
        paste0("AC=2;AF=0.250;AN=8;BaseQRankSum=1.19;DP=72;ExcessHet=0.6695;FS=5.632;MLEAC=2;MLEAF=0.250;",
               "MQ=60.00;MQRankSum=0.00;QD=19.13;ReadPosRankSum=0.410;SOR=0.138;",
               "ANN=C|missense_variant|MODERATE|exon-NM_001144091.1-1|GENE_exon-NM_001144091.1-1|transcript|rna-NM_001144091.1|protein_coding|1/1|c.1241G>C|p.Gly414Ala|1300/2207|1241/1725|414/574||,",
               "C|non_coding_transcript_variant|MODIFIER|FZD7|gene-FZD7|transcript|NM_001144091.1|protein_coding||||||||WARNING_TRANSCRIPT_NO_START_CODON"),
        "GT:AD:DP:GQ:PGT:PID:PL:PS",
        "0|1:8,8:16:99:0|1:90572567_G_T:312,0,312:90572567", "0|1:8,8:16:99:0|1:90572567_G_T:312,0,290:90572567",
        "0/0:20,0:20:48:.:.:0,48,720", "0/0:18,0:18:54:.:.:0,54,445"),
      collapse = "\t")

In [5]:
vcf_lines <- c(vcf_header, rec_1, rec_2)
length(vcf_lines)

[1] 11

**Header vs body**: split on the leading `#`.

In [6]:
meta_lines <- vcf_lines[str_starts(vcf_lines, "##")]
body_lines <- vcf_lines[!str_starts(vcf_lines, "##")]   # header line + records

length(meta_lines)     # meta-information
length(body_lines) - 1 # number of variants

[1] 8

[1] 2

### Parse the records into a tibble

In [7]:
vcf_tbl <- read.delim(text = paste(body_lines, collapse = "\n"), header = TRUE, quote = "",
                      comment.char = "", check.names = FALSE, colClasses = "character") %>%
  as_tibble() %>%
  rename(CHROM = `#CHROM`) %>%
  mutate(POS = as.integer(POS), QUAL = as.numeric(QUAL))

samples <- c("milca_40995", "milca_40998", "control_41011", "control_41012")
select(vcf_tbl, CHROM, POS, REF, ALT, QUAL, FORMAT, all_of(samples))

CHROM,POS,REF,ALT,QUAL,FORMAT,milca_40995,milca_40998,control_41011,control_41012
<chr>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
NC_037329.1,90000329,C,A,226.62,GT:AD:DP:GQ:PL,"0/0:2,0:2:6:0,6,49","0/0:16,0:16:17:0,17,315","0/1:10,6:16:99:100,0,199","0/1:12,8:20:99:138,0,237"
NC_037329.1,90572569,G,C,612.00,GT:AD:DP:GQ:PGT:PID:PL:PS,"0|1:8,8:16:99:0|1:90572567_G_T:312,0,312:90572567","0|1:8,8:16:99:0|1:90572567_G_T:312,0,290:90572567","0/0:20,0:20:48:.:.:0,48,720","0/0:18,0:18:54:.:.:0,54,445"


### The `INFO` column

`INFO` is a semicolon-separated list of `KEY=VALUE` pairs (plus flags without value). Metrics of the **variant as a whole** (QUAL, MQ, QD, FS...) live here, whatever the number of samples.

In [8]:
# Parse an INFO string into a two-column tibble (key, value)
# info: length-1 character, e.g. "AC=2;AF=0.250;DP=54"
# return tibble with columns key (chr) and value (chr)
parse_info <- function(info) {
  stopifnot(is.character(info), length(info) == 1)
  tibble(kv = str_split_1(info, ";")) %>%
    separate(kv, into = c("key", "value"), sep = "=", extra = "merge", fill = "right")
}

`parse_info()` applied to record 2 (the `ANN` key is dropped here)

In [9]:
parse_info(vcf_tbl$INFO[2]) %>% filter(key != "ANN")

key,value
<chr>,<chr>
AC,2
AF,0.250
AN,8
BaseQRankSum,1.19
DP,72
ExcessHet,0.6695
FS,5.632
MLEAC,2
MLEAF,0.250


### The `FORMAT` column and the sample columns

Each sample column stores its values in the order given by `FORMAT`: `GT:AD:DP:GQ:...`

* `GT` genotype (`0/0` hom. ref, `0/1` het., `1/1` hom. alt; `|` instead of `/` means **phased**)
* `AD` allelic depths (ref,alt), `DP` read depth, `GQ` genotype quality (Phred)

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; color: #856404; padding: 10px; border-radius: 4px;">
    ⚠️ <strong>Warning</strong><br>
    <code>FORMAT</code> can <b>differ between records</b>: here the second record also carries <code>PGT:PID:PS</code> (phasing information). Always look fields up <b>by name</b>, never by position.
</div>

In [10]:
# Extract one FORMAT field (by name) from sample strings
# sample_str: character vector of sample columns, e.g. "0/1:10,6:16:99:100,0,199"
# format_str: character vector (same length) of FORMAT strings, e.g. "GT:AD:DP:GQ:PL"
# key: single field name, e.g. "DP"
# return character vector (NA when the key is absent)
get_fmt <- function(sample_str, format_str, key) {
  map2_chr(sample_str, format_str, function(s, f) {
    idx <- match(key, str_split_1(f, ":"))
    if (is.na(idx)) NA_character_ else str_split_1(s, ":")[idx]
  })
}

### Reading all samples with `get_fmt()`

In [11]:
gt_long <- vcf_tbl %>%
  select(CHROM, POS, REF, ALT, FORMAT, all_of(samples)) %>%
  pivot_longer(all_of(samples), names_to = "Indiv", values_to = "raw") %>%
  mutate(GT = get_fmt(raw, FORMAT, "GT"),
         AD = get_fmt(raw, FORMAT, "AD"),
         DP = as.integer(get_fmt(raw, FORMAT, "DP")),
         GQ = as.integer(get_fmt(raw, FORMAT, "GQ")))
select(gt_long, POS, Indiv, GT, AD, DP, GQ)

POS,Indiv,GT,AD,DP,GQ
<int>,<chr>,<chr>,<chr>,<int>,<int>
90000329,milca_40995,0/0,"2,0",2,6
90000329,milca_40998,0/0,"16,0",16,17
90000329,control_41011,0/1,"10,6",16,99
90000329,control_41012,0/1,"12,8",20,99
90572569,milca_40995,0|1,"8,8",16,99
90572569,milca_40998,0|1,"8,8",16,99
90572569,control_41011,0/0,"20,0",20,48
90572569,control_41012,0/0,"18,0",18,54


### The `ANN` field (SnpEff)

`ANN` holds **one annotation per alternative allele and per transcript**, separated by `,`. Each annotation has 16 sub-fields separated by `|`.

In [12]:
ann_fields <- c("allele", "effect", "impact", "gene_name", "gene_id", "feature_type", "feature_id",
                "biotype", "rank", "hgvs_c", "hgvs_p", "cdna_pos", "cds_pos", "aa_pos", "distance", "messages")

# Split the SnpEff ANN field into one row per annotation
# tbl: tibble with columns var_id (chr) and ANN (chr)
# return tibble with var_id and the 16 ANN sub-fields (chr)
parse_ann <- function(tbl) {
  stopifnot(is.data.frame(tbl), all(c("var_id", "ANN") %in% names(tbl)))
  tbl %>%
    distinct(var_id, ANN) %>%
    separate_rows(ANN, sep = ",") %>%
    separate(ANN, into = ann_fields, sep = "\\|", extra = "drop", fill = "right")
}

### Applying `parse_ann()` to the two records

In [13]:
recs <- vcf_tbl %>%
  mutate(var_id = paste(CHROM, POS, REF, ALT, sep = "_"),
         ANN = str_extract(INFO, "(?<=ANN=)[^;]+"))

ann_long <- parse_ann(recs)
ann_long %>% select(var_id, effect, impact, gene_name, feature_id, biotype, hgvs_p)

var_id,effect,impact,gene_name,feature_id,biotype,hgvs_p
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
NC_037329.1_90000329_C_A,intron_variant,MODIFIER,exon-NM_001206233.1-1,rna-NM_001206233.1,protein_coding,
NC_037329.1_90000329_C_A,non_coding_transcript_variant,MODIFIER,TRAK2,NM_001206233.1,protein_coding,
NC_037329.1_90572569_G_C,missense_variant,MODERATE,exon-NM_001144091.1-1,rna-NM_001144091.1,protein_coding,p.Gly414Ala
NC_037329.1_90572569_G_C,non_coding_transcript_variant,MODIFIER,FZD7,NM_001144091.1,protein_coding,


<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; color: #856404; padding: 10px; border-radius: 4px;">
    ⚠️ <strong>Warning</strong><br>
    Look at <code>gene_name</code>: for the <code>missense_variant</code> annotation it is an <b>exon placeholder</b> (<code>exon-NM_...</code>) because the SnpEff database was built by hand from a GFF file. The gene symbol only appears in the <b>other</b> annotation of the same transcript.
</div>

In [14]:
# Recover gene symbols when SnpEff reports exon/transcript placeholders
# ann_long (tibble): returned by parse_ann() (needs gene_name and feature_id)
# placeholder: regex identifying placeholder gene names
# return ann_long plus a column `gene` (chr)
resolve_gene_names <- function(ann_long, placeholder = "^(exon|cds|rna)-") {
    
  stopifnot(is.data.frame(ann_long), all(c("gene_name", "feature_id") %in% names(ann_long)))
    
  tx2gene <- ann_long %>%
    filter(!is.na(gene_name), !str_detect(gene_name, placeholder)) %>%
    mutate(tx = str_remove(feature_id, "^rna-"), gene_symbol = gene_name, .keep="none") %>%
    distinct(tx, .keep_all = TRUE)
  ann_long %>%
    mutate(tx = str_remove(feature_id, "^rna-")) %>%
    left_join(tx2gene, by = "tx") %>%
    mutate(gene = coalesce(gene_symbol, gene_name)) %>%
    select(-tx, -gene_symbol)
}

### Resolving the gene symbols

In [15]:
ann_long <- resolve_gene_names(ann_long)
ann_long %>% select(var_id, effect, gene, hgvs_p)

var_id,effect,gene,hgvs_p
<chr>,<chr>,<chr>,<chr>
NC_037329.1_90000329_C_A,intron_variant,TRAK2,
NC_037329.1_90000329_C_A,non_coding_transcript_variant,TRAK2,
NC_037329.1_90572569_G_C,missense_variant,FZD7,p.Gly414Ala
NC_037329.1_90572569_G_C,non_coding_transcript_variant,FZD7,


## III - Filtering variants: how to proceed?

Many filters can be applied to a VCF:

* **quality filters** (thresholds are somewhat arbitrary): depth, genotype (`0/1`, `1/1`), allelic ratio... ➜ reduce the false-positive rate
* **type of variants** to keep (SNVs only, Indels...)
* **region of interest**

**The rest depends on the biological question**: additional filters, statistical analyses, graphical representations...

### False positives vs quality filters

<div style="display:flex; gap:10px; font-size:0.9em;">
<div style="flex:1; background:#f0f2f5; border-radius:8px; padding:10px;"><strong>Variant calling results</strong><br>true positives (polymorphisms, mutations) + <b>false positives</b>; some true variants may be missed (false negatives)</div>
<div style="align-self:center; font-size:1.4em;">➜ filters / thresholds ➜</div>
<div style="flex:1; background:#dafbe1; border-radius:8px; padding:10px;"><strong>Final result</strong><br>maximum of true variants<br>minimum of false positives</div>
</div>

**The more stringent the filter, the more false positives are removed, but the higher the risk of losing true variants.**

### Reminder: where do false positives come from?

* Artefacts from **PCR cycles** during library preparation
* Artefacts linked to the **sequencing technology** (PacBio, HiSeq, NextSeq...)
* **Alignment difficulties** (repeated DNA regions)
* **Base-calling** errors

### GATK Best Practices: hard filters on INFO metrics

Recommendations based on GATK-specific metrics, **different for SNVs and Indels**. Values below are the thresholds a variant must **pass**.

| Metric | Meaning | SNV: keep if | Indel: keep if |
|---|---|---|---|
| **QD** QualByDepth | QUAL divided by the depth of the samples carrying the variant | > 2 | > 2 |
| **FS** FisherStrand | Phred-scaled p-value of a Fisher exact test for **strand bias** | < 60 | < 200 |
| **SOR** StrandOddsRatio | another strand-bias score, more robust for high-depth sites | < 3 | [Parameter to verify] |
| **MQ** RMSMappingQuality | root mean square of the mapping quality of the reads | > 40 | not used |
| **MQRankSum** | rank-sum test: mapping quality of alt vs ref reads | > -12.5 | not used |
| **ReadPosRankSum** | rank-sum test: position of the alleles along the reads | > -8 | > -20 |

They are starting points, not statistical decision rules: check the distributions on your own data.

In [16]:
# Flag variants failing GATK hard filters 
# tbl: tibble with var_type (chr: "SNV"/"INDEL") and numeric QD, FS, SOR, MQ, MQRankSum, ReadPosRankSum
# return tbl plus logical `hard_fail` (NA metrics never make a variant fail)
add_hard_filter_flag <- function(tbl) {
  stopifnot(is.data.frame(tbl),
            all(c("var_type", "QD", "FS", "SOR", "MQ", "MQRankSum", "ReadPosRankSum") %in% names(tbl)))
  mutate(tbl, hard_fail = case_when(
    var_type == "SNV"   ~ replace_na(QD < 2 | FS > 60 | SOR > 3 | MQ < 40 |
                                     MQRankSum < -12.5 | ReadPosRankSum < -8, FALSE),
    var_type == "INDEL" ~ replace_na(QD < 2 | FS > 200 | ReadPosRankSum < -20, FALSE)))
}

### Testing the function on toy values

In [17]:
toy <- tribble(
  ~id, ~var_type, ~QD,  ~FS,  ~SOR, ~MQ, ~MQRankSum, ~ReadPosRankSum,
  "s1", "SNV",    19.1,  5.6, 0.14, 60,   0,         0.41,   # passes
  "s2", "SNV",     1.2,  3.0, 0.50, 60,   0,         0.00,   # QD too low
  "s3", "SNV",    12.0, 75.0, 4.10, 60,  -1,         0.30,   # strand bias (FS, SOR)
  "s4", "SNV",    15.0,  2.0, 1.00, 32, -13,         0.00,   # low mapping quality
  "s5", "INDEL",  11.0,210.0, 2.00, 60,  NA,        -1.00,   # strand bias (FS)
  "s6", "INDEL",   3.0, 20.0, 6.00, 55,  NA,        NA)      # NA ignored: passes

add_hard_filter_flag(toy)

id,var_type,QD,FS,SOR,MQ,MQRankSum,ReadPosRankSum,hard_fail
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
s1,SNV,19.1,5.6,0.14,60,0,0.41,FALSE
s2,SNV,1.2,3.0,0.50,60,0,0.00,TRUE
s3,SNV,12.0,75.0,4.10,60,-1,0.30,TRUE
s4,SNV,15.0,2.0,1.00,32,-13,0.00,TRUE
s5,INDEL,11.0,210.0,2.00,60,NA,-1.00,TRUE
s6,INDEL,3.0,20.0,6.00,55,NA,NA,FALSE


### Genotype-level filters: depth, GQ and allele balance

`QUAL` and the `INFO` metrics describe the **variant**; `DP`, `GQ` and `AD` describe **one sample at one position**.

**Why a minimum depth?** For a true heterozygote the number of alt reads follows a Binomial(DP, 0.5). With few reads, a heterozygote can look homozygous by chance: the probability of seeing only one allele is `2 * 0.5^DP`.

In [18]:
tibble(DP = c(2, 4, 8, 10, 20)) %>%
  mutate(p_het_looks_hom = 2 * dbinom(0, size = DP, prob = 0.5))

DP,p_het_looks_hom
<dbl>,<dbl>
2,5.000000e-01
4,1.250000e-01
8,7.812500e-03
10,1.953125e-03
20,1.907349e-06


**Allele balance** = alt / (ref + alt). About 0.5 is expected for a heterozygote, close to 1 for a homozygous alternative.

In [19]:
gt_long %>%
  filter(GT != "0/0") %>%
  separate(AD, into = c("ad_ref", "ad_alt"), sep = ",", convert = TRUE) %>%
  mutate(allele_balance = ad_alt / (ad_ref + ad_alt)) %>%
  select(POS, Indiv, GT, DP, allele_balance)

POS,Indiv,GT,DP,allele_balance
<int>,<chr>,<chr>,<int>,<dbl>
90000329,control_41011,0/1,16,0.375
90000329,control_41012,0/1,20,0.400
90572569,milca_40995,0|1,16,0.500
90572569,milca_40998,0|1,16,0.500


### Genotype classes and segregation with the phenotype

Genotypes may be unphased (`0/1`) or phased (`0|1`). We map them to three classes, so that the rest of the code does not depend on the separator.

In [20]:
# Classify biallelic diploid genotypes
# gt (character vector): GT values, e.g. c("0/0", "0|1", "1/1", NA)
# return character vector in {"hom_ref", "het", "hom_alt", NA}
classify_gt <- function(gt) {
  case_when(gt %in% c("0/0", "0|0")                ~ "hom_ref",
            gt %in% c("0/1", "1/0", "0|1", "1|0")  ~ "het",
            gt %in% c("1/1", "1|1")                ~ "hom_alt",
            TRUE                                    ~ NA_character_)
}

### Selecting the variants that segregate with the phenotype

In [21]:
# Select variants segregating with the phenotype
# gt_wide (tibble): var_id + one genotype-class column per sample
# carriers (character vector): columns of the phenotype carriers
# controls (character vector): columns of the controls
# return rows of gt_wide with an alt allele in all carriers and hom_ref in all controls
find_segregating <- function(gt_wide, carriers, controls) {
  stopifnot(is.data.frame(gt_wide), is.character(carriers), is.character(controls),
            all(c(carriers, controls) %in% names(gt_wide)))
  filter(gt_wide,
         if_all(all_of(carriers), ~ .x %in% c("het", "hom_alt")),
         if_all(all_of(controls), ~ .x %in% "hom_ref"))
}

carriers <- c("milca_40995", "milca_40998")
controls <- c("control_41011", "control_41012")

In [22]:
gt_wide <- gt_long %>%
  mutate(var_id = paste(CHROM, POS, REF, ALT, sep = "_"), gt_class = classify_gt(GT)) %>%
  pivot_wider(id_cols = var_id, names_from = Indiv, values_from = gt_class)
gt_wide

var_id,milca_40995,milca_40998,control_41011,control_41012
<chr>,<chr>,<chr>,<chr>,<chr>
NC_037329.1_90000329_C_A,hom_ref,hom_ref,het,het
NC_037329.1_90572569_G_C,het,het,hom_ref,hom_ref


In [23]:
find_segregating(gt_wide, carriers, controls)

var_id,milca_40995,milca_40998,control_41011,control_41012
<chr>,<chr>,<chr>,<chr>,<chr>
NC_037329.1_90572569_G_C,het,het,hom_ref,hom_ref


The first record (`C>A`, TRAK2 region) is heterozygous in the **controls** only, the second one (`G>C`) in the **milca animals** only: only the second one segregates with the phenotype.

## IV - Practical 1: search for a mutation in the milca phenotype

Through the [ONAB](https://www.onab.fr/), breeders reported a difference in coat colour in some Montbéliarde cattle: the **milca phenotype**.

We have the variant calling results of **4 animals** (annotated multi-sample VCF), sequenced on a 1.14 Mb region of chromosome 2 (`NC_037329.1:90000000-91140000`, ARS-UCD2.0).

| Sample | Group |
|---|---|
| milca_40995 | milca (carrier) |
| milca_40998 | milca (carrier) |
| control_41011 | control |
| control_41012 | control |

<div style="background-color: #fbefff; border-left: 6px solid #8250df; padding: 10px;">
    ❓ <strong>Question</strong><br>
    <ul>
<li>Which <b>type of mutation</b> is involved in the milca phenotype in animal 40995?</li>
<li>What is its <b>genotype</b>?</li>
<li>On which <b>gene</b> is it located?</li>
<li>What about the other animals?</li>
</ul>
</div>

**Strategy**

1. Read the VCF into R and convert it to a tibble (`vcfR`).
2. Keep bi-allelic variants, build a unique variant ID, classify genotypes (using previous written functions).
3. Apply quality filters (QUAL < 30, DP < 4, GQ < 20 and with a classified genotypes)
4. Parse the `ANN` field: keep **protein-coding** and **missense** annotations.
5. Keep variants with an alternative allele in **both carriers** and homozygous reference in **both controls**.

The cells below need the `vcfR` package and the annotated VCF of the practical. The helper functions defined above are reused.

In [30]:
suppressMessages(library(vcfR))

vcf_file <- "4samples_selected_region_annotated.vcf"
stopifnot(file.exists(vcf_file))

vcf_obj  <- read.vcfR(vcf_file, verbose = FALSE)

tidy_lst <- suppressMessages(vcfR2tidy(vcf_obj))

map(tidy_lst, head, n=2)

$fix
# A tibble: 2 × 29
  ChromKey CHROM       POS ID    REF   ALT    QUAL FILTER VariantKey AC    AF   
     <int> <chr>     <int> <chr> <chr> <chr> <dbl> <chr>       <int> <chr> <chr>
1        1 NC_0373… 9.00e7 NA    C     A     227.  NA              1 2     0.250
2        1 NC_0373… 9.00e7 NA    T     C      72.8 NA              2 1     0.125
# ℹ 18 more variables: AN <int>, BaseQRankSum <dbl>, DP <int>, END <int>,
#   ExcessHet <dbl>, FS <dbl>, InbreedingCoeff <dbl>, MLEAC <chr>, MLEAF <chr>,
#   MQ <dbl>, MQRankSum <dbl>, QD <dbl>, RAW_MQandDP <chr>,
#   ReadPosRankSum <dbl>, SOR <dbl>, ANN <chr>, LOF <chr>, NMD <chr>

$gt
# A tibble: 2 × 16
  ChromKey      POS VariantKey Indiv    gt_AD gt_DP gt_GQ gt_GT gt_MIN_DP gt_PGT
     <int>    <int>      <int> <chr>    <chr> <int> <int> <chr>     <int> <chr> 
1        1 90000329          1 control… 10,6     16    99 0/1          NA NA    
2        1 90000582          2 control… 8,0       8    24 0/0          NA NA    
# ℹ 6 more variables: gt_PID <chr>, gt_PL <chr>, gt_PS <int>, gt_RGQ <int>,
#   gt_SB <chr>, gt_GT_alleles <chr>

$meta
# A tibble: 2 × 5
  Tag   ID    Number Type    Description                                        
  <chr> <chr> <chr>  <chr>   <chr>                                              
1 INFO  AC    A      Integer Allele count in genotypes, for each ALT allele, in…
2 INFO  AF    A      Float   Allele Frequency, for each ALT allele, in the same…

In [25]:
tp <- full_join(tidy_lst$fix, tidy_lst$gt, by = c("ChromKey", "POS"))

tp %>% head(n=2)

dim(tp)

ChromKey,CHROM,POS,ID,REF,ALT,QUAL,FILTER,VariantKey.x,AC,⋯,gt_GQ,gt_GT,gt_MIN_DP,gt_PGT,gt_PID,gt_PL,gt_PS,gt_RGQ,gt_SB,gt_GT_alleles
<int>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>,<chr>,⋯,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>
1,NC_037329.1,90000329,NA,C,A,226.62,NA,1,2,⋯,99,0/1,NA,NA,NA,"100,0,199",NA,NA,NA,C/A
1,NC_037329.1,90000329,NA,C,A,226.62,NA,1,2,⋯,99,0/1,NA,NA,NA,"138,0,237",NA,NA,NA,C/A


[1] 26612    43

In [26]:
min_qual <- 30; min_dp <- 4; min_gq <- 20

tp <- tp %>%
  filter(!str_detect(ALT, ",")) %>%                                   # bi-allelic only
  mutate(var_id   = paste(CHROM, POS, REF, ALT, sep = "_"),
         var_type = if_else(str_length(REF) == 1 & str_length(ALT) == 1, "SNV", "INDEL"),
         gt_class = classify_gt(gt_GT)) %>%
  filter(QUAL >= min_qual, gt_DP >= min_dp, gt_GQ >= min_gq, !is.na(gt_class))

n_distinct(tp$var_id) # similar to length(unique(tp$var_id))

[1] 6345

In [27]:
ann_tp <- parse_ann(tp) %>% resolve_gene_names()

missense <- ann_tp %>% filter(biotype == "protein_coding", str_detect(effect, "missense_variant"))

candidates <- tp %>%
  filter(var_id %in% missense$var_id) %>%
  select(var_id, Indiv, gt_class) %>%
  pivot_wider(id_cols = var_id, names_from = Indiv, values_from = gt_class) %>%
  find_segregating(carriers, controls)
candidates

var_id,control_41011,control_41012,milca_40995,milca_40998
<chr>,<chr>,<chr>,<chr>,<chr>
NC_037329.1_90572569_G_C,hom_ref,hom_ref,het,het


In [28]:
tp %>% filter(var_id %in% candidates$var_id) %>% 
   select(var_id, Indiv, gt_GT, gt_class)

ann_tp %>% filter(var_id %in% candidates$var_id & effect == "missense_variant")


var_id,Indiv,gt_GT,gt_class
<chr>,<chr>,<chr>,<chr>
NC_037329.1_90572569_G_C,control_41011,0/0,hom_ref
NC_037329.1_90572569_G_C,control_41012,0/0,hom_ref
NC_037329.1_90572569_G_C,milca_40995,0|1,het
NC_037329.1_90572569_G_C,milca_40998,0|1,het


var_id,allele,effect,impact,gene_name,gene_id,feature_type,feature_id,biotype,rank,hgvs_c,hgvs_p,cdna_pos,cds_pos,aa_pos,distance,messages,gene
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
NC_037329.1_90572569_G_C,C,missense_variant,MODERATE,exon-NM_001144091.1-1,GENE_exon-NM_001144091.1-1,transcript,rna-NM_001144091.1,protein_coding,1/1,c.1241G>C,p.Gly414Ala,1300/2207,1241/1725,414/574,,,FZD7


In [29]:
missense %>% filter(var_id %in% candidates$var_id) %>%
  select(var_id, effect, impact, gene, hgvs_c, hgvs_p)

var_id,effect,impact,gene,hgvs_c,hgvs_p
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
NC_037329.1_90572569_G_C,missense_variant,MODERATE,FZD7,c.1241G>C,p.Gly414Ala


### Answer

* **Mutation**: a **SNP** `G>C` at position **90,572,569** of chromosome 2 (`NC_037329.1`).
* **Effect**: **missense variant** (`c.1241G>C`, `p.Gly414Ala`, impact MODERATE) in a protein-coding transcript of **FZD7**.
* **Genotype**: heterozygous **0|1** in both milca animals (40995 and 40998).
* **Other animals**: the two controls (41011 and 41012) are **homozygous reference** (0/0).

<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px;">
    ℹ️ <strong>Note</strong><br>
    With 2 carriers and 2 controls, a perfect segregation cannot be statistically significant (Fisher exact test, smallest attainable p-value 1/3 two-sided). The selection is a <b>deterministic filter</b> combined with biological evidence: the mutation was detected in 3.7 % of Montbéliarde animals and the position is highly conserved among orthologues and paralogues of the FZD gene family (<a href="https://doi.org/10.1111/age.13055">doi:10.1111/age.13055</a>).
</div>

## Summary

**From fastq to VCF, then to biology**

<div style="display:flex; flex-wrap:wrap; align-items:stretch; gap:6px; font-size:0.85em;">
<div style="background:#f0f2f5; border:2px solid #9aa5b1; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Reads</strong><br><span style="font-size:0.9em">FASTQ<br>+ reference genome (FASTA)</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#f0f2f5; border:2px solid #9aa5b1; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Alignment</strong><br><span style="font-size:0.9em">BWA-MEM2<br>➜ BAM</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#f0f2f5; border:2px solid #9aa5b1; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Post-alignment processing<br>&amp; QC</strong><br><span style="font-size:0.9em">GATK MarkDuplicates<br>samtools, mosdepth, qualimap</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#dafbe1; border:2px solid #1f883d; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Variant calling</strong><br><span style="font-size:0.9em">GATK HaplotypeCaller<br>➜ VCF</span></div>
<div style="align-self:center; font-size:1.4em;">➜</div>
<div style="background:#dafbe1; border:2px solid #1f883d; border-radius:8px; padding:8px 10px; min-width:130px; text-align:center;"><strong>Annotation</strong><br><span style="font-size:0.9em">SnpEff / SnpSift<br>➜ annotated VCF</span></div>
</div>

* A VCF = **meta-information + header + one line per variant**; `INFO` describes the variant, `FORMAT` + sample columns describe each genotype. Read fields **by name**.
* The `ANN` field holds several annotations per variant: split it into one row per annotation before filtering.
* Filter on **variant-level** metrics (QUAL, QD, FS, SOR, MQ...) and on **genotype-level** metrics (DP, GQ, allele balance); stringency is a trade-off between false positives and lost true variants.
* Which filters and analyses to run **depends on the biological question**.